### Loading Data

In [20]:
%%writefile train_ddp.py

# Included so the script can be called
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group
import os

# Define model
# Our original Neural Network from the file
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        # Original Sequential model with Linear and Rectilinear layers
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    # Flattens and stacks
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

class Trainer:
    # Original __init__
    def __init__(self, model: torch.nn.Module, train_data: DataLoader, optimizer: torch.optim.Optimizer, gpu_id: int, save_every: int) -> None:
        self.gpu_id = gpu_id
        self.model = model.to(gpu_id)
        self.train_data = train_data
        self.optimizer = optimizer
        self.save_every = save_every

    # Original _run_batch
    def _run_batch(self, source, targets):
        self.optimizer.zero_grad()
        source = source.view(source.size(0), -1)
        output = self.model(source)
        loss = F.cross_entropy(output, targets)
        loss.backward()
        self.optimizer.step()

    def _run_epoch(self, epoch):
        b_sz = len(next(iter(self.train_data))[0])
        # Line was added for visualizing Epochs
        print(f"[GPU{self.gpu_id}] Epoch {epoch} | Batchsize: {b_sz} | Steps: {len(self.train_data)}")
        for source, targets in self.train_data:
            source = source.to(self.gpu_id)
            targets = targets.to(self.gpu_id)
            self._run_batch(source, targets)

    # Original _save_checkpoint
    def _save_checkpoint(self, epoch):
        ckp = self.model.state_dict()
        PATH = "checkpoint.pt"
        torch.save(ckp, PATH)
        print(f"Epoch {epoch} | Training checkpoint saved at {PATH}")


    def train(self, max_epochs: int, test_data: DataLoader | None = None):
        # We define a loss function so we can perform validation
        loss_fn = torch.nn.CrossEntropyLoss()
        
        for epoch in range(max_epochs):
            self._run_epoch(epoch)
            # Modified so we are only saving the checkpoint
            # On the initial GPU
            if self.gpu_id == 0 and epoch % self.save_every == 0:
                self._save_checkpoint(epoch)

            # We only perform validation on the origial GPU as well
            if self.gpu_id == 0 and test_data is not None:
                test(test_data, self.model, loss_fn, self.gpu_id)

# Load train objs was 
# Adapted from multi_gpu file
# Here we load data into a train and test set
# Originally the variables were declared outside
# of a function
def load_train_objs():
    train_set = datasets.FashionMNIST(
        root="data",
        train=True,
        download=True,
        transform=ToTensor(),
    )

    test_set = datasets.FashionMNIST(
        root = "data",
        train = False,
        download = True,
        transform = ToTensor()
    )

    # We also adapted the code so it uses the 
    # NeuralNetwork built in this file
    model = NeuralNetwork()
    optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)
    return train_set, test_set, model, optimizer

# Here we have a DataLoader for iterating our data,
# Though we use a DistributedSampler
def prepare_dataloader(dataset: Dataset, batch_size: int, world_size, rank):
    return DataLoader(
        dataset,
        batch_size = batch_size,
        sampler = DistributedSampler(dataset, num_replicas = world_size, rank = rank, shuffle = True),
        pin_memory = True
    )

# Here we define a test function, just like the original file
def test(dataloader, model, loss_fn, gpu_id: int):
    if isinstance(model, torch.nn.parallel.DistributedDataParallel):
        model = model.module

    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    test_loss, correct = 0.0, 0.0
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(gpu_id, non_blocking = True)
            y = y.to(gpu_id, non_blocking = True)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).float().sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error:\n Accuracy {100*correct:>0.1f}% Avg loss: {test_loss:>8f}\n", flush = True)

# This is a little bit different 
# than the original file as we encapsulate
# the logic within a dataloader function
def prepare_test_dataloader(dataset: Dataset, batch_size: int):
    return DataLoader(dataset, batch_size = batch_size, shuffle = False, pin_memory = True)

# This code was adapted from the multi_gpu file
def ddp_setup(rank, world_size):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"
    torch.cuda.set_device(rank)
    init_process_group(backend = 'nccl', rank = rank, world_size = world_size)

# Here we:
#  1) call ddp_setup, just like in multi_gpu
#  2) build a train and test dataloder
#  3) call the trainer identified in this file
#  4) train using the mixture of this file and adapted multi-gpu code
#  5) destroy_process_group() is called to prevent crashing
def main(rank: int, world_size: int, save_every: int, total_epochs: int, batch_size: int):
    print(f"[rank {rank}] starting (world_size=({world_size})", flush = True)
    ddp_setup(rank, world_size)
    try:
        train_set, test_set, model, optimizer = load_train_objs()
        train_dataloader = prepare_dataloader(train_set, batch_size, world_size, rank)
        test_dataloader = prepare_test_dataloader(test_set, batch_size)

        trainer = Trainer(model, train_dataloader, optimizer, rank, save_every)
        trainer.train(total_epochs, test_data = test_dataloader)
    finally:
        destroy_process_group()

# Here we spawn the code just like in the multi_gpu file
if __name__ == "__main__":
    world_size = torch.cuda.device_count()
    mp.spawn(main, args = (world_size, 5, 5, 64), nprocs = world_size)

Overwriting train_ddp.py


In [21]:
!python -u train_ddp.py

[rank 1] starting (world_size=(2)
[rank 0] starting (world_size=(2)
[GPU1] Epoch 0 | Batchsize: 64 | Steps: 469
[GPU0] Epoch 0 | Batchsize: 64 | Steps: 469
Epoch 0 | Training checkpoint saved at checkpoint.pt
[GPU1] Epoch 1 | Batchsize: 64 | Steps: 469
Test Error:
 Accuracy 52.3% Avg loss: 2.226912

[GPU0] Epoch 1 | Batchsize: 64 | Steps: 469
[GPU1] Epoch 2 | Batchsize: 64 | Steps: 469
Test Error:
 Accuracy 57.0% Avg loss: 2.141269

[GPU0] Epoch 2 | Batchsize: 64 | Steps: 469
[GPU1] Epoch 3 | Batchsize: 64 | Steps: 469
Test Error:
 Accuracy 56.0% Avg loss: 2.022328

[GPU0] Epoch 3 | Batchsize: 64 | Steps: 469
[GPU1] Epoch 4 | Batchsize: 64 | Steps: 469
Test Error:
 Accuracy 56.3% Avg loss: 1.859607

[GPU0] Epoch 4 | Batchsize: 64 | Steps: 469
Test Error:
 Accuracy 58.2% Avg loss: 1.668007

